# Projeto Fase 2 - Billboard Hot 100

Notebook PySpark para preparação dos dados, criação de variáveis por música e aplicação de aprendizagem não supervisionada com PCA e K-Means.


## 1. Setup

Criação da sessão Spark e definição de parâmetros gerais para garantir reprodutibilidade.


In [4]:
import os
from pathlib import Path

os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler, PCA
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

RANDOM_SEED = 42
PCA_COMPONENTS = 5
K_VALUES = range(2, 9)

spark = (
    SparkSession.builder
    .appName("Billboard_Fase2")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark UI:", spark.sparkContext.uiWebUrl)


Spark UI: http://localhost:4040


In [5]:
# O enunciado recomenda /BigData/Projecto/data/.
# Para trabalhar localmente, se esse caminho nao existir, usa-se a pasta atual do notebook.
data_dir = Path("/BigData/Projecto/data")
if not data_dir.exists():
    data_dir = Path.cwd()

output_dir = Path.cwd() / "outputs_fase2"
output_dir.mkdir(exist_ok=True)

print("Pasta dos dados:", data_dir)
print("Pasta de outputs:", output_dir)


Pasta dos dados: /content
Pasta de outputs: /content/outputs_fase2


## 2. Leitura dos dados

A leitura é feita sem inferência automática de tipos para evitar problemas com valores `NA` e campos textuais complexos. Os tipos são corrigidos depois.


In [8]:
csv_reader = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("sep", ",")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", True)
    .option("mode", "PERMISSIVE")
)

df_af_raw = csv_reader.csv("/audio_features.csv")
df_bb_raw = csv_reader.csv("/billboard.csv")

print("audio_features:", df_af_raw.count(), "linhas e", len(df_af_raw.columns), "colunas")
print("billboard:", df_bb_raw.count(), "linhas e", len(df_bb_raw.columns), "colunas")

df_af_raw.printSchema()
df_bb_raw.printSchema()


audio_features: 29503 linhas e 22 colunas
billboard: 327895 linhas e 10 colunas
root
 |-- song_id: string (nullable = true)
 |-- performer: string (nullable = true)
 |-- song: string (nullable = true)
 |-- spotify_genre: string (nullable = true)
 |-- spotify_track_id: string (nullable = true)
 |-- spotify_track_preview_url: string (nullable = true)
 |-- spotify_track_duration_ms: string (nullable = true)
 |-- spotify_track_explicit: string (nullable = true)
 |-- spotify_track_album: string (nullable = true)
 |-- danceability: string (nullable = true)
 |-- energy: string (nullable = true)
 |-- key: string (nullable = true)
 |-- loudness: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- speechiness: string (nullable = true)
 |-- acousticness: string (nullable = true)
 |-- instrumentalness: string (nullable = true)
 |-- liveness: string (nullable = true)
 |-- valence: string (nullable = true)
 |-- tempo: string (nullable = true)
 |-- time_signature: string (nullable = tru

## 3. Funções auxiliares de limpeza


In [9]:
AUDIO_NUMERIC_COLS = [
    "spotify_track_duration_ms", "danceability", "energy", "key", "loudness", "mode",
    "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "tempo",
    "time_signature", "spotify_track_popularity"
]

BILLBOARD_NUMERIC_COLS = [
    "week_position", "instance", "previous_week_position", "peak_position", "weeks_on_chart"
]

MODEL_FEATURE_COLS = [
    "spotify_track_duration_ms", "danceability", "energy", "key", "loudness", "mode",
    "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "tempo",
    "time_signature", "spotify_track_popularity", "chart_weeks", "best_position",
    "worst_position", "avg_position", "position_volatility", "position_range",
    "entry_position", "final_position", "num_instances", "reentry_count",
    "top10_weeks", "top10_share", "time_to_peak_weeks"
]


def replace_na_strings(df):
    return df.select([
        F.when(F.trim(F.col(c).cast("string")) == "NA", None)
         .when(F.trim(F.col(c).cast("string")) == "", None)
         .otherwise(F.col(c))
         .alias(c)
        for c in df.columns
    ])


def cast_numeric_columns(df, cols):
    for c in cols:
        df = df.withColumn(c, F.expr(f"try_cast(`{c}` as double)"))
    return df


def missing_counts(df):
    return df.select([
        F.count(F.when(
            F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), c
        )).alias(c)
        for c in df.columns
    ])


## 4. Diagnóstico inicial de qualidade dos dados


In [10]:
print("Valores em falta em audio_features, incluindo NA textual:")
missing_counts(replace_na_strings(df_af_raw)).show(truncate=False)

print("Valores em falta em billboard, incluindo NA textual:")
missing_counts(replace_na_strings(df_bb_raw)).show(truncate=False)

print("Duplicados exatos em audio_features:", df_af_raw.count() - df_af_raw.dropDuplicates().count())
print("Duplicados exatos em billboard:", df_bb_raw.count() - df_bb_raw.dropDuplicates().count())


Valores em falta em audio_features, incluindo NA textual:
+-------+---------+-----+-------------+----------------+-------------------------+-------------------------+----------------------+-------------------+------------+------+-----+--------+-----+-----------+------------+----------------+--------+-------+-----+--------------+--------------------------+
|song_id|performer|song |spotify_genre|spotify_track_id|spotify_track_preview_url|spotify_track_duration_ms|spotify_track_explicit|spotify_track_album|danceability|energy|key  |loudness|mode |speechiness|acousticness|instrumentalness|liveness|valence|tempo|time_signature|spotify_track_popularity;;|
+-------+---------+-----+-------------+----------------+-------------------------+-------------------------+----------------------+-------------------+------------+------+-----+--------+-----+-----------+------------+----------------+--------+-------+-----+--------------+--------------------------+
|0      |23239    |23239|24709        |260

## 5. Limpeza e correção de tipos

Nesta etapa são substituídos os `NA` textuais por `null`, removidos duplicados e convertidas as variáveis numéricas.


In [14]:
# 1. Limpeza inicial e remoção de duplicados
df_af = replace_na_strings(df_af_raw).dropDuplicates()
df_bb = replace_na_strings(df_bb_raw).dropDuplicates()

# 2. CORREÇÃO DE NOMES: Remove os ";;" que vieram do CSV
df_af = df_af.withColumnRenamed("spotify_track_popularity;;", "spotify_track_popularity")
df_bb = df_bb.withColumnRenamed("weeks_on_chart;;", "weeks_on_chart")

# 3. Conversão de tipos (agora os nomes batem com suas listas AUDIO_NUMERIC_COLS e BILLBOARD_NUMERIC_COLS)
df_af = cast_numeric_columns(df_af, AUDIO_NUMERIC_COLS)
df_bb = cast_numeric_columns(df_bb, BILLBOARD_NUMERIC_COLS)

# 4. Criar a coluna de data
df_bb = df_bb.withColumn("week_date", F.to_date("week_id", "M/d/yyyy"))

# Verificação
print("audio_features limpo:", df_af.count(), "linhas")
print("billboard limpo:", df_bb.count(), "linhas")

df_af.select("song_id", "performer", "song", "danceability", "energy", "tempo").show(5, truncate=False)
df_bb.select("song_id", "week_date", "week_position", "previous_week_position", "peak_position", "weeks_on_chart").show(5, truncate=False)

audio_features limpo: 29479 linhas
billboard limpo: 327895 linhas
+------------------------------------------------------+----------------+------------------------------------------+------------+------+-------+
|song_id                                               |performer       |song                                      |danceability|energy|tempo  |
+------------------------------------------------------+----------------+------------------------------------------+------------+------+-------+
|Do You Miss MeJocelyn Enriquez                        |Jocelyn Enriquez|Do You Miss Me                            |0.72        |0.872 |129.956|
|Love Gets RoughTroy Newman                            |Troy Newman     |Love Gets Rough                           |0.56        |0.665 |95.013 |
|NoDodie Stevens                                       |Dodie Stevens   |No                                        |0.692       |0.424 |135.403|
|Sweet Sensual LoveBig Mountain                        |Big Moun

## 6. Unidade de análise: música (`song_id`)

O ficheiro `billboard.csv` é longitudinal: uma música aparece várias semanas. Para clustering, a unidade de análise passa a ser a música, agregada por `song_id`.


In [15]:
song_window_asc = Window.partitionBy("song_id").orderBy(F.col("week_date").asc())
song_window_desc = Window.partitionBy("song_id").orderBy(F.col("week_date").desc())

first_rows = (
    df_bb.withColumn("rn_first", F.row_number().over(song_window_asc))
    .filter(F.col("rn_first") == 1)
    .select(
        "song_id",
        F.col("week_date").alias("first_week_date"),
        F.col("week_position").alias("entry_position")
    )
)

last_rows = (
    df_bb.withColumn("rn_last", F.row_number().over(song_window_desc))
    .filter(F.col("rn_last") == 1)
    .select(
        "song_id",
        F.col("week_date").alias("last_week_date"),
        F.col("week_position").alias("final_position")
    )
)

peak_rows = (
    df_bb.groupBy("song_id")
    .agg(F.min("week_position").alias("best_position"))
    .join(df_bb.select("song_id", "week_date", "week_position"), on="song_id", how="inner")
    .filter(F.col("week_position") == F.col("best_position"))
    .groupBy("song_id", "best_position")
    .agg(F.min("week_date").alias("peak_week_date"))
)

bb_song = df_bb.groupBy("song_id").agg(
    F.first("song", ignorenulls=True).alias("song"),
    F.first("performer", ignorenulls=True).alias("performer"),
    F.count("*").cast("double").alias("chart_weeks"),
    F.max("week_position").alias("worst_position"),
    F.avg("week_position").alias("avg_position"),
    F.stddev("week_position").alias("position_volatility"),
    F.max("instance").alias("num_instances"),
    F.sum(F.when(F.col("week_position") <= 10, 1).otherwise(0)).cast("double").alias("top10_weeks"),
    F.max("weeks_on_chart").alias("max_weeks_on_chart")
)

bb_song = (
    bb_song
    .join(first_rows, on="song_id", how="left")
    .join(last_rows, on="song_id", how="left")
    .join(peak_rows, on="song_id", how="left")
    .withColumn("position_range", F.col("worst_position") - F.col("best_position"))
    .withColumn("reentry_count", F.greatest(F.col("num_instances") - F.lit(1), F.lit(0.0)))
    .withColumn("top10_share", F.col("top10_weeks") / F.col("chart_weeks"))
    .withColumn("time_to_peak_weeks", F.datediff(F.col("peak_week_date"), F.col("first_week_date")) / F.lit(7.0))
)

print("Numero de musicas distintas:", bb_song.count())
bb_song.select(
    "song_id", "song", "performer", "chart_weeks", "best_position",
    "entry_position", "final_position", "top10_share", "time_to_peak_weeks"
).show(10, truncate=False)


Numero de musicas distintas: 27903
+-------------------------------------------------+-----------------------------+---------------------------------+-----------+-------------+--------------+--------------+-------------------+------------------+
|song_id                                          |song                         |performer                        |chart_weeks|best_position|entry_position|final_position|top10_share        |time_to_peak_weeks|
+-------------------------------------------------+-----------------------------+---------------------------------+-----------+-------------+--------------+--------------+-------------------+------------------+
|NULL                                             |NULL                         |NULL                             |16333.0    |NULL         |NULL          |NULL          |0.0                |NULL              |
|#1 Dee JayGoody Goody                            |#1 Dee Jay                   |Goody Goody                      |5.0   

## 7. Junção com características de áudio

As características do Spotify são adicionadas à tabela agregada por música através da chave `song_id`.


In [16]:
af_song = df_af.dropDuplicates(["song_id"]).select(
    "song_id", "spotify_genre", "spotify_track_explicit", "spotify_track_album", *AUDIO_NUMERIC_COLS
)

model_df = bb_song.join(af_song, on="song_id", how="left")

print("Dataset final antes da imputacao:", model_df.count(), "linhas e", len(model_df.columns), "colunas")
missing_counts(model_df.select(*MODEL_FEATURE_COLS)).show(truncate=False)


Dataset final antes da imputacao: 27903 linhas e 37 colunas
+-------------------------+------------+------+-----+--------+-----+-----------+------------+----------------+--------+-------+-----+--------------+------------------------+-----------+-------------+--------------+------------+-------------------+--------------+--------------+--------------+-------------+-------------+-----------+-----------+------------------+
|spotify_track_duration_ms|danceability|energy|key  |loudness|mode |speechiness|acousticness|instrumentalness|liveness|valence|tempo|time_signature|spotify_track_popularity|chart_weeks|best_position|worst_position|avg_position|position_volatility|position_range|entry_position|final_position|num_instances|reentry_count|top10_weeks|top10_share|time_to_peak_weeks|
+-------------------------+------------+------+-----+--------+-----+-----------+------------+----------------+--------+-------+-----+--------------+------------------------+-----------+-------------+-------------

## 8. Imputação, normalização e PCA

Os valores em falta nas variáveis de modelação são imputados pela mediana. Depois as variáveis são normalizadas e reduzidas por PCA.


In [17]:
imputed_cols = [f"{c}_imp" for c in MODEL_FEATURE_COLS]

imputer = Imputer(
    inputCols=MODEL_FEATURE_COLS,
    outputCols=imputed_cols,
    strategy="median"
)

assembler = VectorAssembler(inputCols=imputed_cols, outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", withStd=True, withMean=True)
pca = PCA(k=PCA_COMPONENTS, inputCol="scaled_features", outputCol="pca_features")

prep_pipeline = Pipeline(stages=[imputer, assembler, scaler, pca])
prep_model = prep_pipeline.fit(model_df)
pca_df = prep_model.transform(model_df).cache()

pca_model = prep_model.stages[-1]
explained_variance = [float(v) for v in pca_model.explainedVariance]

print("Variancia explicada por componente:")
for i, v in enumerate(explained_variance, start=1):
    print(f"PC{i}: {v:.4f}")
print(f"Total explicado pelas {PCA_COMPONENTS} componentes: {sum(explained_variance):.4f}")


Variancia explicada por componente:
PC1: 0.1826
PC2: 0.1055
PC3: 0.0822
PC4: 0.0812
PC5: 0.0660
Total explicado pelas 5 componentes: 0.5176


## 9. Escolha do número de clusters

São testados vários valores de `k`. A escolha é apoiada pelo silhouette e pela interpretabilidade dos grupos.


In [18]:
evaluator = ClusteringEvaluator(
    featuresCol="pca_features",
    predictionCol="prediction",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

k_results = []
best_model = None
best_k = None
best_score = None

for k in K_VALUES:
    km = KMeans(
        featuresCol="pca_features",
        predictionCol="prediction",
        k=k,
        seed=RANDOM_SEED,
        maxIter=50
    )
    model = km.fit(pca_df)
    pred = model.transform(pca_df)
    silhouette = evaluator.evaluate(pred)
    inertia = float(model.summary.trainingCost)
    k_results.append((int(k), float(silhouette), float(inertia)))
    print(f"k={k}: silhouette={silhouette:.4f}; inertia={inertia:.2f}")

    if best_score is None or silhouette > best_score:
        best_score = silhouette
        best_k = k
        best_model = model

print(f"Melhor k por silhouette: {best_k} ({best_score:.4f})")

print("\nResumo dos valores testados:")
print("k | silhouette | inertia")
for k, silhouette, inertia in k_results:
    print(f"{k} | {silhouette:.4f} | {inertia:.2f}")


k=2: silhouette=0.8041; inertia=371732.51
k=3: silhouette=0.6205; inertia=282983.85
k=4: silhouette=0.5450; inertia=193856.30
k=5: silhouette=0.5547; inertia=149433.96
k=6: silhouette=0.4555; inertia=126638.81
k=7: silhouette=0.5409; inertia=92501.82
k=8: silhouette=0.5681; inertia=84039.97
Melhor k por silhouette: 2 (0.8041)

Resumo dos valores testados:
k | silhouette | inertia
2 | 0.8041 | 371732.51
3 | 0.6205 | 282983.85
4 | 0.5450 | 193856.30
5 | 0.5547 | 149433.96
6 | 0.4555 | 126638.81
7 | 0.5409 | 92501.82
8 | 0.5681 | 84039.97


## 10. Modelo final K-Means


In [19]:
cluster_df = best_model.transform(pca_df).cache()

print("Dimensao dos clusters:")
cluster_sizes = cluster_df.groupBy("prediction").count().orderBy("prediction")
cluster_sizes.show()


Dimensao dos clusters:
+----------+-----+
|prediction|count|
+----------+-----+
|         0|26704|
|         1| 1199|
+----------+-----+



## 11. Perfil dos clusters

Esta tabela é a principal base para interpretar os grupos no relatório.


In [20]:
profile_cols = [
    "chart_weeks", "best_position", "avg_position", "position_volatility",
    "entry_position", "final_position", "top10_share", "time_to_peak_weeks",
    "danceability", "energy", "valence", "tempo", "spotify_track_popularity"
]

cluster_profile = cluster_df.groupBy("prediction").agg(
    *[F.round(F.avg(f"{c}_imp"), 3).alias(c) for c in profile_cols]
).orderBy("prediction")

cluster_profile.show(truncate=False)


+----------+-----------+-------------+------------+-------------------+--------------+--------------+-----------+------------------+------------+------+-------+-------+------------------------+
|prediction|chart_weeks|best_position|avg_position|position_volatility|entry_position|final_position|top10_share|time_to_peak_weeks|danceability|energy|valence|tempo  |spotify_track_popularity|
+----------+-----------+-------------+------------+-------------------+--------------+--------------+-----------+------------------+------------+------+-------+-------+------------------------+
|0         |11.77      |46.351       |61.757      |15.61              |80.214        |76.308        |0.054      |6.741             |0.604       |0.593 |0.678  |118.077|0.03                    |
|1         |11.328     |53.734       |68.467      |13.937             |85.393        |85.157        |0.031      |6.758             |0.706       |0.746 |0.74   |119.789|0.031                   |
+----------+-----------+------

## 12. Exemplos de músicas por cluster

São selecionadas músicas representativas com maior permanência no ranking e melhor posição.


In [21]:
examples_window = Window.partitionBy("prediction").orderBy(
    F.col("chart_weeks_imp").desc(),
    F.col("best_position_imp").asc()
)

cluster_examples = (
    cluster_df.withColumn("rn", F.row_number().over(examples_window))
    .filter(F.col("rn") <= 8)
    .select(
        "prediction", "song", "performer",
        F.round("chart_weeks_imp", 0).alias("chart_weeks"),
        F.round("best_position_imp", 0).alias("best_position"),
        F.round("avg_position_imp", 2).alias("avg_position"),
        F.round("top10_share_imp", 3).alias("top10_share")
    )
    .orderBy("prediction", "rn")
)

cluster_examples.show(100, truncate=False)


+----------+---------------------+-----------------------------------------+-----------+-------------+------------+-----------+
|prediction|song                 |performer                                |chart_weeks|best_position|avg_position|top10_share|
+----------+---------------------+-----------------------------------------+-----------+-------------+------------+-----------+
|0         |NULL                 |NULL                                     |16333.0    |46.0         |64.0        |0.0        |
|0         |Radioactive          |Imagine Dragons                          |87.0       |3.0          |32.82       |0.23       |
|0         |Sail                 |AWOLNATION                               |79.0       |17.0         |50.41       |0.0        |
|0         |Blinding Lights      |The Weeknd                               |76.0       |1.0          |9.95        |0.75       |
|0         |I'm Yours            |Jason Mraz                               |76.0       |6.0          |29

## 13. Observações potencialmente atípicas

Como alternativa simples à deteção formal de anomalias, calculamos a distância de cada música ao centróide do seu cluster no espaço PCA. As músicas mais distantes são candidatas a casos atípicos.


In [22]:
from pyspark.ml.functions import vector_to_array

centers = best_model.clusterCenters()

# Evita UDF Python: convertemos o vetor PCA para array e calculamos a distancia
# com expressoes nativas do Spark, mais estaveis no Windows.
anomalies_df = cluster_df.withColumn("pca_array", vector_to_array(F.col("pca_features")))

distance_expr = None
for cluster_id, center in enumerate(centers):
    cluster_distance = None
    for i, value in enumerate(center):
        term = F.pow(F.col("pca_array").getItem(i) - F.lit(float(value)), 2)
        cluster_distance = term if cluster_distance is None else cluster_distance + term

    condition = F.col("prediction") == F.lit(cluster_id)
    distance_expr = (
        F.when(condition, cluster_distance)
        if distance_expr is None
        else distance_expr.when(condition, cluster_distance)
    )

anomalies_df = anomalies_df.withColumn("distance_to_centroid", distance_expr)

possible_anomalies = anomalies_df.select(
    "song", "performer", "prediction",
    F.round("distance_to_centroid", 3).alias("distance_to_centroid"),
    F.round("chart_weeks_imp", 0).alias("chart_weeks"),
    F.round("best_position_imp", 0).alias("best_position"),
    F.round("position_volatility_imp", 3).alias("position_volatility")
).orderBy(F.col("distance_to_centroid").desc())

possible_anomalies.show(20, truncate=False)


+-------------------------------------------------------+----------------------------------------------------+----------+--------------------+-----------+-------------+-------------------+
|song                                                   |performer                                           |prediction|distance_to_centroid|chart_weeks|best_position|position_volatility|
+-------------------------------------------------------+----------------------------------------------------+----------+--------------------+-----------+-------------+-------------------+
|Over The Mountain; Across The Sea                      |Johnnie & Joe                                       |1         |66312.527           |2.0        |89.0         |2.121              |
|Ain't Got No; I Got Life                               |Nina Simone                                         |1         |45044.212           |4.0        |94.0         |2.16               |
|Rockin' Around The Christmas Tree                     

## 14. Guardar outputs para o relatório

Os resultados principais são guardados em CSV na pasta `outputs_fase2`.


In [23]:
import csv


def write_rows_to_csv(rows, file_path):
    rows = list(rows)
    if not rows:
        return
    columns = list(rows[0].asDict().keys())
    with open(file_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        for row in rows:
            writer.writerow(row.asDict())


# Evitamos DataFrame.write.csv no Windows porque pode exigir HADOOP_HOME/winutils.
k_results_path = output_dir / "k_results.csv"
with open(k_results_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["k", "silhouette", "inertia"])
    writer.writerows(k_results)

write_rows_to_csv(cluster_sizes.collect(), output_dir / "cluster_sizes.csv")
write_rows_to_csv(cluster_profile.collect(), output_dir / "cluster_profile.csv")
write_rows_to_csv(cluster_examples.collect(), output_dir / "cluster_examples.csv")
write_rows_to_csv(possible_anomalies.limit(50).collect(), output_dir / "possible_anomalies.csv")

print("Outputs guardados em:", output_dir)
print("-", k_results_path.name)
print("- cluster_sizes.csv")
print("- cluster_profile.csv")
print("- cluster_examples.csv")
print("- possible_anomalies.csv")


Outputs guardados em: /content/outputs_fase2
- k_results.csv
- cluster_sizes.csv
- cluster_profile.csv
- cluster_examples.csv
- possible_anomalies.csv


## 15. Notas para interpretação no relatório

Depois de executar o notebook, devem ser usados sobretudo:

- a tabela de comparação de `k`;
- a variância explicada pela PCA;
- o perfil médio dos clusters;
- os exemplos de músicas por cluster;
- a lista de observações potencialmente atípicas.

A interpretação final deve transformar os clusters em perfis substantivos, por exemplo: êxitos imediatos, músicas de longa permanência, músicas com pico curto, slow burners ou músicas com características sonoras atípicas.
